# Agente Analista Eleitoral 2026
**Disciplina:** Agentes Cognitivos com LangChain (Infnet) — **Autor:** David Mercado

Agente que responde perguntas sobre a eleição para **Deputado Federal de 2026 em SP (70 vagas) e BA (39 vagas)**, combinando:
dados oficiais do TSE (DuckDB), uma calculadora determinística do sistema proporcional, RAG sobre a legislação eleitoral e um verificador da resposta.

| Fase | Conteúdo |
|---|---|
| 0 | Ambiente e dados |
| 1 | LLM isolado (linha de base) |
| 2 | Chain MVP com saída estruturada; 2b: parser, LCEL × @chain, roteamento, paralelo |
| 3 | Ferramentas determinísticas |
| 4 | RAG sobre a legislação |
| 5 | Agente com memória |
| 6 | Agente integrador com verificação, callbacks e executores clássicos |
| 7 | Avaliação de respostas e de trajetória (log próprio e LangSmith) |
| 8 | Interface Streamlit |

Kernel: `.venv` da pasta do projeto. LM Studio ligado com `qwen/qwen3-14b` e `text-embedding-bge-m3`.

## Arquitetura cognitiva

**Anatomia do agente**

| Componente | Implementação |
|---|---|
| Percepção (entrada) | pergunta em linguagem natural; intenção e entidades extraídas (Fase 2b) |
| Raciocínio / planejamento | ciclo **ReAct** do `create_agent`: pensar → escolher ferramenta → observar → repetir; prompt PTCF com passo a passo (intenção → entidades → ferramenta → conferência); limite de 10 chamadas |
| Memória | trabalho (mensagens), episódica (checkpointer SQLite por `thread_id`), semântica (banco do TSE + legislação via FAISS), procedimental (prompt) |
| Ação (ferramentas) | 11 ferramentas: SQL, calculadora, rankings, RAG, artigo exato, API do IBGE, finanças |
| Verificação / autocorreção | `auditor.py` confere a resposta; se reprovada, o agente replaneja e reescreve uma vez |

**Tipo:** agente único, *tool-augmented*, padrão ReAct com *function calling*, com verificador externo determinístico (não é multiagente).

**Nível de maturidade agêntica** (escala usada aqui): 1) LLM isolado → 2) chain fixa → 3) chain com roteamento → 4) agente que escolhe ferramentas e tem memória → **5) agente com verificação e autocorreção** → 6) multiagente / planejamento hierárquico autônomo. O projeto está no **nível 5**: decide quais ferramentas usar, mantém contexto e corrige a própria resposta quando a verificação falha; não cria planos longos nem coordena outros agentes.

**Planejamento e memória aumentada:** o planejamento é reativo (ReAct), passo a passo, com replanejamento disparado pelo verificador. A memória é aumentada por recuperação: o LLM não guarda fatos eleitorais; eles vêm do banco e do índice vetorial, e o histórico longo é resumido (Fase 5).

**Por que LangChain/LangGraph**

| Orquestrador | Ponto forte | Por que não foi o escolhido aqui |
|---|---|---|
| **LangChain 1.x + LangGraph** | ferramentas, RAG, SQL, memória (checkpointer), middleware, LangSmith integrados; roda com LM Studio via API OpenAI | **escolhido**: cobre o agente único com ferramentas heterogêneas e a avaliação |
| LlamaIndex | ingestão e consulta de documentos (RAG) | forte em RAG, mais fraco em agente com SQL, cálculo e memória por conversa |
| CrewAI | equipes de agentes com papéis | o problema não precisa de vários agentes; mais chamadas ao LLM local, que é lento |
| AutoGen (Microsoft) | conversa entre agentes, geração de código | idem; foco multiagente |
| Semantic Kernel | integração com ecossistema .NET/Azure | ecossistema menos alinhado ao Python/DuckDB do projeto |


## Fase 0 — Ambiente e dados

In [59]:
import os, sys, re
from pathlib import Path

# garante que o notebook rode a partir da pasta do projeto
raiz = Path.cwd()
if not (raiz / "config.py").exists():
    achados = list(raiz.rglob("Agente Eleitoral/config.py"))
    assert achados, "Abra o notebook dentro da pasta 'Agente Eleitoral'"
    raiz = achados[0].parent
os.chdir(raiz); sys.path.insert(0, str(raiz))

import config, verificar_ambiente
verificar_ambiente.verificar()

Python 3.14.3 em c:\Users\david\OneDrive\Documentos\Estudo_atual\Mestrado\Agentes Cognitivos com LangChain\Agente Eleitoral\.venv\Scripts\python.exe
  [ok]    duckdb
  [ok]    pandas
  [ok]    dotenv
  [ok]    langchain
  [ok]    langchain_openai
  [ok]    langchain_community
  [ok]    langchain_text_splitters
  [ok]    langgraph
  [ok]    langgraph.checkpoint.sqlite
  [ok]    langsmith
  [ok]    faiss
  [ok]    pypdf
  [ok]    pytest
  [ok]    ipykernel
  [ok]    langchain_classic
  [ok]    streamlit
Dados TSE (4): consulta_cand_2026.zip, consulta_vagas_2026.zip, votacao_candidato_munzona_2026.zip, votacao_partido_munzona_2026.zip
Legislação: 6 PDF(s)
Banco DuckDB: ok
Índice FAISS: ok
LM Studio: ok em http://192.168.15.8:1234/v1
  - qwen/qwen3-14b <- agente
  - text-embedding-bge-m3 <- embedding
  - text-embedding-nomic-embed-text-v1.5
  - meta-llama-3-8b-instruct
  - google/gemma-3-12b <- juiz
  - openai/gpt-oss-20b


True

In [60]:
import db
con = db.conectar()   # somente leitura
con.sql("""SELECT v.uf, v.vagas,
       (SELECT COUNT(*) FROM candidatos c WHERE c.uf = v.uf) AS candidatos,
       (SELECT SUM(votos_validos) FROM votos_agremiacao_uf a WHERE a.uf = v.uf) AS votos_validos
FROM vagas v ORDER BY 1""").df()

,uf,vagas,candidatos,votos_validos
0,BA,39,535,8281816.0
1,SP,70,1132,23674188.0


### Validação da calculadora contra o resultado oficial
A calculadora (`calculadora.py`) implementa quociente eleitoral, quociente partidário (mínimo de 10% do QE) e sobras (barreira 80/20 e fase final do STF). Antes de entregar isso ao agente, ela é conferida com a situação oficial de cada candidato no TSE.

In [61]:
for uf in config.UFS:
    resultado, comp = db.recalcular_uf(con, uf)
    print(resultado.resumo())
    print(f"-> {uf}: {comp['coincidentes']}/{comp['eleitos_oficiais']} eleitos coincidem com o TSE | idêntico: {comp['acerto_total']}\n")

Vagas: 70 | Votos válidos: 23.674.188 | QE: 338.203
  PL: 19 cadeira(s) (QP = 19)
  FEDERAÇÃO BRASIL DA ESPERANÇA - FE BRASIL: 11 cadeira(s) (QP = 8)
  FEDERAÇÃO PSOL REDE: 6 cadeira(s) (QP = 9)
  PODE: 6 cadeira(s) (QP = 5)
  PSD: 6 cadeira(s) (QP = 4)
  REPUBLICANOS: 6 cadeira(s) (QP = 4)
  FEDERAÇÃO UNIÃO PROGRESSISTA: 5 cadeira(s) (QP = 5)
  MDB: 4 cadeira(s) (QP = 3)
  NOVO: 2 cadeira(s) (QP = 1)
  PSB: 2 cadeira(s) (QP = 2)
  FEDERAÇÃO PSDB CIDADANIA: 1 cadeira(s) (QP = 1)
  MISSÃO: 1 cadeira(s) (QP = 1)
  FEDERAÇÃO RENOVAÇÃO SOLIDÁRIA: 1 cadeira(s) (QP = 1)
-> SP: 70/70 eleitos coincidem com o TSE | idêntico: True

Vagas: 39 | Votos válidos: 8.281.816 | QE: 212.354
  FEDERAÇÃO BRASIL DA ESPERANÇA - FE BRASIL: 10 cadeira(s) (QP = 9)
  FEDERAÇÃO UNIÃO PROGRESSISTA: 8 cadeira(s) (QP = 7)
  PSD: 6 cadeira(s) (QP = 5)
  PL: 4 cadeira(s) (QP = 3)
  AVANTE: 3 cadeira(s) (QP = 2)
  REPUBLICANOS: 3 cadeira(s) (QP = 3)
  FEDERAÇÃO PSDB CIDADANIA: 2 cadeira(s) (QP = 2)
  MDB: 1 cadeira(s) 

## Fase 1 — LLM isolado (linha de base)
O mesmo modelo, sem ferramentas. Ele não tem os dados de 2026: ou admite que não sabe ou inventa. É o problema que o projeto resolve.

In [62]:
llm = config.llm()
limpar = lambda t: re.sub(r"<think>.*?</think>", "", t, flags=re.DOTALL).strip()

pergunta_vitrine = "Por que o Rui Falcão não foi eleito deputado federal em SP em 2026?"
print(limpar(llm.invoke(pergunta_vitrine + " /no_think").content))

Até agora, em 2024, ainda não ocorreu as eleições federais de 2026. As eleições para deputados federais no Brasil são realizadas a cada quatro anos, geralmente nos meses de outubro, com o pleito de 2026 sendo o próximo após as eleições de 2022.

Portanto, não há informações disponíveis sobre por que Rui Falcão (ou qualquer outro candidato) não foi eleito em 2026, pois esse evento ainda não ocorreu. Se você estiver se referindo a uma situação hipotética ou a um cenário futuro, seria necessário considerar fatores como:

- Resultados eleitorais locais e nacionais;
- Estratégias de campanha;
- Suporte político e popularidade do candidato;
- Concorrência com outros candidatos da mesma legenda ou de partidos rivais;
- Eventos políticos ou sociais que possam ter influenciado o eleitorado.

Se você tiver mais contexto ou quiser saber sobre a trajetória política de Rui Falcão, posso ajudar também!


## Fase 2 — Chain MVP com saída estruturada
Primeira chain: `prompt | llm.with_structured_output(...)`. O modelo classifica a pergunta em uma categoria fechada (`Literal`), o que permite decidir em código o que fazer com ela.

In [63]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

class Classificacao(BaseModel):
    categoria: Literal["dados", "calculo", "legislacao", "financas", "fora_de_escopo"]
    uf: Literal["SP", "BA", "nenhuma"] = Field(description="UF citada na pergunta")
    justificativa: str = Field(description="uma frase")

prompt = ChatPromptTemplate.from_messages([
    ("system", "Classifique perguntas sobre a eleição de Deputado Federal 2026 em SP e BA. "
               "Outras UFs, cargos ou anos são fora_de_escopo. /no_think"),
    ("human", "{pergunta}")])
classificador = prompt | llm.with_structured_output(Classificacao)

for p in ["Quem foi o mais votado na Bahia?", "O que diz o art. 108 do Código Eleitoral?",
          "Quem ganhou para governador no Rio?"]:
    print(p, "->", classificador.invoke({"pergunta": p}))

Quem foi o mais votado na Bahia? -> categoria='calculo' uf='BA' justificativa='A pergunta pede o candidato mais votado na Bahia, que se enquadra em uma análise de resultados eleitorais.'
O que diz o art. 108 do Código Eleitoral? -> categoria='fora_de_escopo' uf='nenhuma' justificativa='A pergunta refere-se ao conteúdo de um artigo do Código Eleitoral, que não está relacionado diretamente à eleição de Deputado Federal em 2026 nas Unidades Federativas de São Paulo (SP) e Bahia (BA).'
Quem ganhou para governador no Rio? -> categoria='fora_de_escopo' uf='nenhuma' justificativa='A pergunta refere-se à eleição para governador no Rio de Janeiro, que não está dentro do escopo definido (Deputado Federal 2026 em SP e BA).'


**Observação:** a saída estruturada funciona (sempre uma categoria válida do `Literal`), mas a classificação nem sempre é certa — na execução gravada, "O que diz o art. 108 do Código Eleitoral?" foi classificada como `fora_de_escopo`. Uma chain de uma etapa não basta; daí as ferramentas e a verificação das fases seguintes.

## Fase 2b — Output parser, LCEL × @chain, Runnable, roteamento e paralelismo (`chains.py`)
**Reconhecimento de intenção e extração de entidades** em JSON validado por `PydanticOutputParser`. Tratamento de erro: se o JSON vier inválido, o erro volta ao LLM uma vez; se falhar de novo, a pergunta é tratada como fora de escopo (falha segura).

In [64]:
import chains
print(chains.parser.get_format_instructions()[:600])

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "Intenção e entidades de uma pergunta (reconhecimento de intenção + extração de entidades).", "properties": {"intencao": {"enum": ["e


In [65]:
classificador = chains.criar_classificador()
# Runnable.invoke
print(classificador.invoke({"pergunta": "Por que o Rui Falcão não foi eleito em SP?"}))
# Runnable.batch: várias perguntas numa chamada
for r in classificador.batch([{"pergunta": "Quais os mais votados em Luís Eduardo Magalhães?"},
                              {"pergunta": "O que diz o art. 108 do Código Eleitoral?"},
                              {"pergunta": "Quem ganhou para governador no Rio?"}], config={"max_concurrency": 1}):
    print(r)

intencao='resultado_uf' uf='SP' candidato='Rui Falcão' municipio=None artigo=None
intencao='ranking_municipio' uf='BA' candidato=None municipio='Luís Eduardo Magalhães' artigo=None
intencao='legislacao' uf='nenhuma' candidato=None municipio=None artigo='108'
intencao='fora_de_escopo' uf='nenhuma' candidato=None municipio=None artigo=None


**Roteamento dinâmico com `RunnableBranch`** (equivalente atual do RouterChain): a intenção escolhe a ferramenta determinística. **Composição declarativa (LCEL)**: `classificar | rotear | redigir`, uma chain sequencial.

In [66]:
pipeline_lcel = chains.criar_pipeline_lcel()
print(pipeline_lcel.invoke({"pergunta": "Quais os 3 mais votados na Bahia?"}))

1. PASTOR SARGENTO ISIDÓRIO (AVANTE) — 248.147 votos — situação: ELEITO POR QP  
2. ANTONIO BRITO (PSD) — 205.815 votos — situação: ELEITO POR QP  
3. NETO CARLETTO (AVANTE) — 197.240 votos — situação: ELEITO POR QP


In [67]:
# Runnable.stream: a resposta chega em pedaços (útil na interface)
for pedaco in pipeline_lcel.stream({"pergunta": "Por que o Rui Falcão não foi eleito em SP?"}):
    print(pedaco, end="", flush=True)

Rui Falcão não foi eleito devido à legislação eleitoral, especificamente ao Código Eleitoral, art. 108, que estabelece a necessidade de candidatos atingirem uma votação mínima de 10% do quociente eleitoral (QE) para serem elegíveis por quociente partidário. Como Rui Falcão não atingiu esse limiar mínimo, ele não foi eleito.

**Observação (execução de 13/11):** aqui a chain sem agente errou o motivo — o classificador marcou a pergunta sobre o Rui Falcão como `resultado_uf`, a ferramenta devolveu o resultado geral e o redator **inventou** que ele não atingiu 10% do QE (ele tinha 31,5%). É o risco de uma chain fixa sem verificação. Correção: quando há candidato citado, o roteador vai direto para `explicar_candidato`.

**Composição imperativa com `@chain`**: a mesma lógica em Python comum, com desvios explícitos (fora de escopo, candidato não encontrado) e retorno da trajetória. Continua sendo um Runnable.

| | Declarativa (LCEL) | Imperativa (`@chain`) |
|---|---|---|
| Forma | `a \| b \| c`, `RunnableParallel`, `RunnableBranch` | função Python decorada |
| Ganha de graça | batch, stream, async, paralelismo, tracing | if/try/laços, regras de negócio, logs no meio |
| Uso no projeto | pipeline fixo classificar → rotear → redigir | fluxo com desvios e falha segura |

Escolha: declarativo quando o caminho é fixo; imperativo quando o caminho depende de decisões em código.

In [68]:
pipeline_imp = chains.criar_pipeline_imperativo()
for p in ["Quantas cadeiras os votos do Lucas Pavanato renderam sozinhos?", "Quem ganhou para governador no Rio?"]:
    r = pipeline_imp.invoke(p)
    print(f"{p}\n  intenção: {r['intencao'].intencao} | resposta: {r['resposta'][:300]}\n")

Quantas cadeiras os votos do Lucas Pavanato renderam sozinhos?
  intenção: explicar_candidato | resposta: Os votos de Lucas Pavanato renderam sozinhos **8 cadeiras** de quociente partidário [Código Eleitoral, arts. 106 a 108].

Quem ganhou para governador no Rio?
  intenção: fora_de_escopo | resposta: Não cubro essa pergunta: trabalho com Deputado Federal 2026 em SP e BA.



In [69]:
# RunnableParallel: as duas UFs recalculadas em paralelo
import time
t0 = time.perf_counter()
ambas = chains.comparar_ufs.invoke(None)
print(f"{time.perf_counter() - t0:.2f}s")
for uf, txt in ambas.items():
    print(uf, "->", txt.splitlines()[0])

0.27s
SP -> Vagas: 70 | Votos válidos: 23.674.188 | QE: 338.203
BA -> Vagas: 39 | Votos válidos: 8.281.816 | QE: 212.354


## Fase 3 — Ferramentas determinísticas
Regra do projeto: **quem calcula é o código; o LLM escolhe a ferramenta e redige**. Cada ferramenta pode ser testada sem LLM.

In [70]:
import tools
for f in tools.FERRAMENTAS:
    print(f"{f.name:22s} {f.description.splitlines()[0][:95]}")

esquema_banco          Lista as views do banco eleitoral (Deputado Federal 2026, SP e BA), suas colunas e exemplos de 
consultar_banco        Executa UMA consulta SQL somente leitura (SELECT ou WITH, dialeto DuckDB) no banco eleitoral e 
resultado_eleicao      Recalcula a distribuição das cadeiras de Deputado Federal de uma UF ('SP' ou 'BA'): votos válid
explicar_candidato     Explica, com os números do cálculo oficial, por que um candidato a Deputado Federal foi ou não 
mais_votados_uf        Ranking ESTADUAL: os candidatos a Deputado Federal mais votados em todo o estado ('SP' ou 'BA')
eleitos_agremiacao     Lista os eleitos de um partido ou federação na UF, do mais ao menos votado, com o critério de e
votos_por_municipio    Ranking dos candidatos a Deputado Federal mais votados em UM MUNICÍPIO (cidade) de SP ou BA — s
perfil_municipio_ibge  Consulta a API do IBGE (externa) e cruza com o TSE: população residente do município no Censo 2
buscar_legislacao      Busca trechos da legislaç

In [71]:
print(tools.explicar_candidato.invoke({"uf": "SP", "candidato": "Rui Falcão"}))

CONCLUSÃO: NÃO ELEITO porque a agremiação conquistou 11 cadeira(s) e a candidatura ficou em 12º lugar na lista interna; as cadeiras vão aos mais votados da agremiação, e o último eleito dela (JOAO PAULO CUNHA) teve 109.548 votos. A candidatura cumpria os mínimos individuais; o que faltou foi vaga para a agremiação.

DADOS:
- Candidato: RUI FALCÃO (FEDERAÇÃO BRASIL DA ESPERANÇA - FE BRASIL), SP. Situação oficial no TSE: SUPLENTE.
- Votos nominais válidos: 106.443 (49º da UF; 12º da agremiação).
- Quociente eleitoral (QE) de SP: 338.203 = 23.674.188 votos válidos ÷ 70 vagas.
- Agremiação: 2.843.542 votos válidos = 8,41 QE -> QP 8; cadeiras no total (QP + sobras): 11.
- Votação individual = 31,5% do QE. Atinge 10% do QE (exigido para vaga por QP)? SIM. Atinge 20% do QE (exigido nas sobras com barreira)? SIM.
- Teve mais votos que SÔNIA GUAJAJARA (FEDERAÇÃO PSOL REDE, 54.207 votos), o eleito menos votado da UF: no sistema proporcional as cadeiras são primeiro das agremiações, depois dos ca

In [72]:
# Text-to-SQL protegido: só SELECT, banco aberto em modo leitura, sem acesso a arquivos externos
print(tools.consultar_banco.invoke({"sql": "SELECT nome_urna, agremiacao, votos FROM ranking_candidato_uf WHERE uf = 'BA' ORDER BY votos DESC LIMIT 3"}))
print(tools.consultar_banco.invoke({"sql": "DROP VIEW candidatos"}))

nome_urna | agremiacao | votos
PASTOR SARGENTO ISIDÓRIO | AVANTE | 248147
ANTONIO BRITO | PSD | 205815
NETO CARLETTO | AVANTE | 197240
(3 linha(s))
ERRO na consulta: Somente consultas SELECT ou WITH são permitidas. Revise o SQL (use esquema_banco para ver as colunas).


In [73]:
# o caso do "puxador de votos"
print(tools.explicar_candidato.invoke({"uf": "SP", "candidato": "Pavanato"}).splitlines()[0])

CONCLUSÃO: ELEITO pelo quociente partidário: a agremiação teve QP 19 e está em 1º lugar na lista da agremiação, dentro das vagas do QP. SOZINHOS, os votos dele valem 8,98 QE = 8 cadeira(s) de quociente partidário (ele ocupa uma; as demais vão aos seguintes da lista). A agremiação inteira conquistou 19.


### Ferramentas hardcoded × semânticas × API externa

| Tipo | Ferramentas | Comportamento |
|---|---|---|
| **Hardcoded (determinísticas)** | `resultado_eleicao`, `explicar_candidato`, `mais_votados_uf`, `eleitos_agremiacao`, `votos_por_municipio`, `analisar_financas` | consultas e cálculos fixos; mesma entrada → mesma saída; testáveis sem LLM |
| **Text-to-query** | `esquema_banco` + `consultar_banco` | o LLM escreve o SQL; o código valida (só SELECT) e executa em modo leitura |
| **Semânticas** | `buscar_legislacao` (FAISS + bge-m3) | recuperação por similaridade; resultado depende do embedding |
| **Busca exata** | `ler_artigo` | índice por artigo, sem embedding |
| **API externa** | `perfil_municipio_ibge` | HTTP na API pública do IBGE (Censo 2022) cruzado com o TSE; falha de rede vira mensagem, não exceção |

APIs internas (DuckDB, índice FAISS) são chamadas diretamente e sempre; a externa (IBGE) só quando a pergunta envolve população, com tempo limite e cache.

In [74]:
print(tools.perfil_municipio_ibge.invoke({"uf": "BA", "municipio": "Luís Eduardo Magalhães"}))

API do IBGE indisponível no momento (UnicodeDecodeError). Tente mais tarde.


## Fase 4 — RAG sobre a legislação
Um chunk por artigo (1.116 trechos de 6 documentos), cada um com a citação pronta. Dois modos de busca: semântica (FAISS) e exata por artigo.

In [75]:
import rag
from collections import Counter
chunks = rag.carregar_chunks()
print(len(chunks), "chunks")
Counter(d.metadata["fonte"] for d in chunks)

1116 chunks


Counter({'Código Eleitoral (Lei 4.737/1965)': 474,
         'Lei das Eleições (Lei 9.504/1997)': 346,
         'Res.-TSE 23.609/2019': 124,
         'STF, ADI 7228': 117,
         'Res.-TSE 23.677/2021': 46,
         'Res.-TSE 23.748/2026': 9})

In [76]:
print(tools.buscar_legislacao.invoke({"pergunta": "o que acontece se nenhum partido atingir o quociente eleitoral?", "k": 2})[:1500])

[Código Eleitoral (Lei 4.737/1965), art. 111] (distância 0.605)
Art. 111. Se nenhum partido alcançar o quociente eleitoral, considerar-se-ão eleitos, até
serem preenchidos todos os lugares, os candidatos mais votados.
Art. 111 com redação dada pelo art. 1º da Lei nº 14.211/2021.
Ac.-STF, de 28/2/2024, nas ADIs n. 7228, 7263 e 7325 declara a inconstitucionalidade deste artigo e
do art. 13 da Res.-TSE n. 23677/2021 para que, no caso de nenhum partido alcançar o quociente
eleitoral, sejam aplicados, sucessivamente, o inciso I c.c. o § 2º e, na sequência, o inciso III do art. 109
do CE/1965, de maneira que a distribuição das cadeiras ocorra, primeiramente, com a aplicação da
cláusula de barreira 80/20, e, quando não houver mais partidos e candidatos que atendam a tal
exigência, as cadeiras restantes sejam distribuídas por média, com a participação de todos os partidos,
ou seja, nos moldes da 3ª fase, sem exigência da cláusula de desempenho 80%, em estrito respeito ao
sistema proporcional.


### Comparação de modelos de embedding
8 perguntas com o artigo esperado. Resultado medido em 09/11 (`3_comparar_embeddings.bat`):

| Modelo | Hit@4 | MRR |
|---|---|---|
| nomic-embed-text-v1.5 (com prefixos) | 75% | 0,59 |
| **bge-m3** | **100%** | **0,88** |

A célula abaixo refaz a medida para o modelo em uso.

In [77]:
vs = rag.carregar_indice()
rag.imprimir_avaliacao(config.MODELO_EMBEDDING, rag.avaliar_recuperacao(vs))


Avaliação da recuperação — text-embedding-bge-m3: Hit@4 = 100% | MRR = 0.88
  ok  pos=1  o que acontece se nenhum partido atingir o quociente eleitoral?
  ok  pos=2  como se calcula o quociente eleitoral?
  ok  pos=2  como é determinado o quociente partidário?
  ok  pos=1  votação nominal mínima de 10% do quociente eleitoral
  ok  pos=1  distribuição das sobras pela maior média com 80% e 20% do quociente
  ok  pos=1  empate na votação entre candidatos do mesmo partido
  ok  pos=1  regras aplicáveis às federações de partidos
  ok  pos=1  critério de desempate entre médias de partidos


## Fase 5 — Agente com memória
`create_agent` (LangChain 1.x) + checkpointer. Taxonomia CoALA: memória de **trabalho** (mensagens da conversa), **episódica** (checkpointer SQLite por `thread_id`), **semântica** (banco do TSE e legislação, via ferramentas) e **procedimental** (regras do prompt).

In [78]:
import agente, uuid
execucao = uuid.uuid4().hex[:6]      # ids novos a cada execução: a memória SQLite não acumula entre rodadas
ag = agente.criar_agente()          # memória persistente em dados/artefatos/memoria_agente.sqlite

def mostrar(r):
    print(r.texto)
    v = "" if r.verificacao is None else f" | verificação: {r.verificacao.resumo()}" + (" (reescrita)" if r.revisada else "")
    print(f"\n[ferramentas: {' -> '.join(r.ferramentas) or 'nenhuma'}{v} | {r.segundos}s]")

r1 = agente.perguntar(ag, "Por que o Rui Falcão não foi eleito em SP?", thread_id=f"demo-memoria-{execucao}")
mostrar(r1)

Rui Falcão não foi eleito porque a agremiação (Federação Brasil da Esperança - FE Brasil) conquistou **11 cadeiras** e ele ficou em **12º lugar na lista interna**. As cadeiras são distribuídas primeiro para os candidatos mais votados de cada agremiação, e o último eleito da agremiação teve **109.548 votos**, superando a quantidade de votos de Rui Falcão (**106.443**).

- Votos nominais válidos: **106.443**.
- Quociente eleitoral (QE) de SP: **338.203**.
- Agremiação: **2.843.542 votos válidos**, correspondendo a **8,41 QE** → QP 8; cadeiras no total (QP + sobras): **11**.
- Votação individual = **31,5% do QE**.

BASE LEGAL aplicável: [Código Eleitoral, arts. 106 a 108]; [Código Eleitoral, art. 109, § 1º]; [Res.-TSE 23.677/2021, art. 11, § 8º].

[ferramentas: explicar_candidato | verificação: ok (reescrita) | 826.2s]


In [79]:
# mesma conversa: "dele" depende do turno anterior
mostrar(agente.perguntar(ag, "E quem ficou com a última vaga da federação dele?", thread_id=f"demo-memoria-{execucao}"))

A última vaga da federação de Rui Falcão foi conquistada por **João Paulo Cunha** (PT), que teve **109.548 votos** e foi eleito por média.

- Último eleito da agremiação: **JOÃO PAULO CUNHA (PT)** — 109.548 votos.
- Primeiros suplentes: RUI FALCÃO (PT) — 106.443 votos; PAULO TEIXEIRA (PT) — 106.062 votos; REIS (PT) — 90.224 votos.

A federação teve **11 cadeiras** no total, sendo **8 por quociente partidário** e **3 por média**.

[ferramentas: eleitos_agremiacao | verificação: ok (reescrita) | 302.5s]


In [80]:
# outra conversa (thread nova): o agente não tem esse contexto
mostrar(agente.perguntar(ag, "E quem ficou com a última vaga da federação dele?", thread_id=f"demo-outra-{execucao}"))

A federação PSOL REDE não obteve eleitos na Bahia, pois o quociente partidário (QP) foi zero. Portanto, não houve uma "última vaga" para essa federação no estado.

[ferramentas: nenhuma | verificação: ok | 51.7s]


**Observação:** na conversa nova o agente não sabe quem é "ele" — a memória é isolada por `thread_id`. Mas, em vez de perguntar, ele escolheu um referente por conta própria (nas execuções gravadas, a Federação PSOL REDE, às vezes sem chamar ferramenta). O verificador não pega, porque a resposta não traz números inventados. É uma limitação registrada no relatório.

### Memória de curto × longo prazo (CoALA) e janela de contexto

| Memória (CoALA) | Prazo | Implementação |
|---|---|---|
| Trabalho | curto | mensagens da conversa no estado do grafo |
| Episódica | longo | `SqliteSaver`: cada conversa (`thread_id`) persiste entre execuções |
| Semântica | longo | fatos eleitorais no DuckDB e legislação no FAISS, acessados por ferramentas |
| Procedimental | longo | regras do prompt (PTCF) e código das ferramentas |

**Janela de contexto:** `SummarizationMiddleware` resume o histórico antigo e mantém só as últimas mensagens; a busca vetorial (FAISS) traz o conhecimento sob demanda, em vez de mantê-lo no contexto. Abaixo, o resumo forçado com um gatilho baixo (4 mensagens).

In [81]:
from langgraph.checkpoint.memory import InMemorySaver
ag_curto = agente.criar_agente(checkpointer=InMemorySaver(), gatilho_resumo=("messages", 4), manter=("messages", 2))
for p in ["Qual o quociente eleitoral de SP?", "E o da Bahia?", "Qual dos dois é maior?"]:
    print(">>", p, "\n", agente.perguntar(ag_curto, p, thread_id=f"janela-{execucao}", verificar=False).texto[:250], "\n")
estado = ag_curto.get_state({"configurable": {"thread_id": f"janela-{execucao}"}}).values["messages"]
print(f"{len(estado)} mensagens na janela; a primeira é o resumo:\n", str(estado[0].content)[:600])

>> Qual o quociente eleitoral de SP? 
 O quociente eleitoral (QE) em São Paulo (SP) foi de **338.203** votos. 

>> E o da Bahia? 
 (o agente não produziu resposta) 

>> Qual dos dois é maior? 
 (o agente não produziu resposta) 

4 mensagens na janela; a primeira é o resumo:
 Here is a summary of the conversation to date:

## SESSION INTENT  
The user is asking for the electoral quotient (QE) of Bahia (BA).  

## SUMMARY  
- The user previously asked for the QE of São Paulo (SP), which was 338,203 votes.  
- For Bahia (BA), the QE is **212,354** votes, calculated from **8,281,816 valid votes** and **39 seats**.  
- Key distribution: FEDERAÇÃO BRASIL DA ESPERANÇA - FE BRASIL secured 10 seats, FEDERAÇÃO UNIÃO PROGRESSISTA 8 seats, PSD 6 seats, PL 4 seats, and others.  
- **6 candidates** were elected via sobras (remainders) using a 80/20 media rule, with FELIX MENDON


### Ferramentas especializadas de memória

| Ferramenta | Arquitetura | Caso de uso | Avaliação para este projeto |
|---|---|---|---|
| **Checkpointer LangGraph** (usado) | estado do grafo serializado por `thread_id` (memória, SQLite, Postgres) | retomar conversas, auditoria | suficiente: conversas curtas, um usuário, tudo local |
| LangGraph Store / LangMem | armazenamento chave-valor com busca semântica, entre conversas | preferências e fatos do usuário | útil se o agente precisasse lembrar o usuário entre conversas |
| Mem0 | camada de memória que extrai e consolida fatos com LLM + vetor/grafo | assistentes pessoais | custo de LLM extra a cada turno; modelo local é lento |
| Zep | serviço com grafo temporal de conhecimento e resumos | memória de longo prazo em produção | exige servidor; excessivo para o escopo |
| Letta (MemGPT) | memória hierárquica que o próprio agente edita (núcleo/arquivo) | agentes de vida longa | complexidade alta; o conhecimento aqui é estável e vem do TSE |
| ConversationBufferMemory (clássico) | histórico inteiro no prompt | chatbots simples | estoura o contexto em conversas longas; demonstrado abaixo |

In [82]:
import legado
bot = legado.chatbot_buffer()
for p in ["Meu nome é David e estudo o sistema proporcional.", "O que é quociente eleitoral, em uma frase?",
          "Qual é o meu nome e o que eu estudo?"]:
    print(">>", p, "\n", legado.limpar(bot.invoke({"input": p})["response"]), "\n")
print("Conteúdo da ConversationBufferMemory:\n", bot.memory.buffer)

>> Meu nome é David e estudo o sistema proporcional. 
 Olá, David! É bom saber que você está estudando o sistema proporcional. No Brasil, esse sistema é usado para eleger deputados federais, estaduais e vereadores, garantindo representatividade proporcional aos votos obtidos pelos partidos. Se quiser, posso explicar como funciona em detalhes. 

>> O que é quociente eleitoral, em uma frase? 
 O quociente eleitoral é o número mínimo de votos necessários para que um candidato seja eleito em um sistema proporcional. 

>> Qual é o meu nome e o que eu estudo? 
 Seu nome é David e você está estudando o sistema proporcional. 

Conteúdo da ConversationBufferMemory:
 Human: Meu nome é David e estudo o sistema proporcional.
AI: Olá, David! É bom saber que você está estudando o sistema proporcional. No Brasil, esse sistema é usado para eleger deputados federais, estaduais e vereadores, garantindo representatividade proporcional aos votos obtidos pelos partidos. Se quiser, posso explicar como funci

## Fase 6 — Agente integrador com verificação
Depois de cada resposta, o `auditor.py` confere (sem LLM): números e citações contra as saídas das ferramentas, coerência com a CONCLUSÃO da ferramenta, links inventados e linguagem de prestação de contas. Resposta reprovada volta ao agente com a lista de problemas e é reescrita uma vez.

Abaixo, duas respostas **reais** do agente nos testes, que o verificador reprova.

In [83]:
import auditor
saida = tools.explicar_candidato.invoke({"uf": "SP", "candidato": "Pavanato"})

resposta_real = ("O Pavanatto não conseguiu nenhuma cadeira sozinho com os votos dele. Ele foi eleito pelo quociente "
                 "partidário, e sua agremiação (PL) obteve 19 cadeiras no total. Votos: 3.038.438.")
for p in auditor.verificar(resposta_real, [saida]).problemas: print("-", p)

saida2 = tools.resultado_eleicao.invoke({"uf": "SP"})
resposta_real2 = ("A eleita menos votada foi Sônia Guajajara, com 54.207 votos, pelo quociente partidário. "
                  "A Federação PSOL Rede obteve 1.386.945 votos válidos, o que equivale a 4,1 QE.")
for p in auditor.verificar(resposta_real2, [saida2]).problemas: print("-", p)

- A resposta omite números da CONCLUSÃO da ferramenta: 8,98. A conclusão da ferramenta deve ser mantida.
- A resposta contradiz a CONCLUSÃO: os votos do candidato sozinhos renderam cadeiras.
- Valores que não aparecem nas saídas das ferramentas: 1.386.945, 4,1.


In [84]:
# agente completo, com verificação
mostrar(agente.perguntar(ag, "Quantas cadeiras os votos do Lucas Pavanato renderam sozinhos?"))

Os votos de Lucas Pavanato renderam sozinhos **8,98 quocientes eleitorais**, correspondendo a **8 cadeiras** no cálculo do quociente partidário.

Base legal: [Código Eleitoral, art. 106 a 108].

[ferramentas: explicar_candidato | verificação: ok (reescrita) | 98.6s]


In [85]:
# legislação + fora de escopo
mostrar(agente.perguntar(ag, "O que diz o art. 108 do Código Eleitoral?"))
print("\n" + "-" * 80)
mostrar(agente.perguntar(ag, "Quem foi o deputado federal mais votado no Rio de Janeiro?"))

O art. 108 do Código Eleitoral estabelece que estarão eleitos, entre os candidatos registrados por um partido que tenham obtido votos em número igual ou superior a 10% (dez por cento) do quociente eleitoral, tantos quantos o respectivo quociente partidário indicar, na ordem da votação nominal que cada um tenha recebido.

[Código Eleitoral (Lei 4.737/1965), art. 108]

[ferramentas: ler_artigo | verificação: ok | 77.2s]

--------------------------------------------------------------------------------
Não cubro essa eleição: trabalho apenas com DEPUTADO FEDERAL 2026 em SÃO PAULO (SP, 70 vagas) e na BAHIA (BA, 39 vagas).

[ferramentas: nenhuma | verificação: ok | 22.9s]


### Callbacks de logging
`callbacks.LogAgente` (um `BaseCallbackHandler`) é passado em cada chamada do agente e registra início, fim e erro de cada ferramenta e de cada chamada ao modelo, com duração, em `evidencias/log_agente.jsonl`. Observa o agente de fora: vale para qualquer ferramenta, sem alterá-la.

In [86]:
import pandas as pd, json
log = pd.DataFrame(json.loads(l) for l in open("evidencias/log_agente.jsonl", encoding="utf-8"))
print(len(log), "eventos")
log[log.evento.str.startswith("ferramenta")][["quando", "sessao", "evento", "ferramenta", "segundos"]].tail(10)

133 eventos


,quando,sessao,evento,ferramenta,segundos
107,2026-11-13T11:27:46,demo-memoria,ferramenta_inicio,eleitos_agremiacao,NaN
108,2026-11-13T11:27:46,demo-memoria,ferramenta_fim,eleitos_agremiacao,0.28
112,2026-11-13T11:31:09,janela,ferramenta_inicio,resultado_eleicao,NaN
113,2026-11-13T11:31:09,janela,ferramenta_fim,resultado_eleicao,0.26
117,2026-11-13T11:33:41,janela,ferramenta_inicio,resultado_eleicao,NaN
118,2026-11-13T11:33:42,janela,ferramenta_fim,resultado_eleicao,0.20
124,2026-11-13T11:41:27,496cc794-ae05-4a7d-a234-2a7bb9b187ea,ferramenta_inicio,explicar_candidato,NaN
125,2026-11-13T11:41:27,496cc794-ae05-4a7d-a234-2a7bb9b187ea,ferramenta_fim,explicar_candidato,0.12
129,2026-11-13T11:42:56,c0893730-2818-4f69-8671-eb370c792f72,ferramenta_inicio,ler_artigo,NaN
130,2026-11-13T11:42:56,c0893730-2818-4f69-8671-eb370c792f72,ferramenta_fim,ler_artigo,0.02


In [87]:
# tempo médio por tipo de evento: o gargalo é o modelo, não as ferramentas
log.groupby(["evento"])["segundos"].agg(["count", "mean"]).round(2)

,count,mean
evento,,
ferramenta_fim,29,0.26
ferramenta_inicio,0,NaN
llm_erro,0,NaN
llm_fim,73,58.09


## Fase 6b — Executores clássicos (`legado.py`)
As mesmas ferramentas com as APIs clássicas, para comparação. O agente principal usa as APIs atuais (`create_agent` + checkpointer), que substituem estas na versão 1.0.

**1. `initialize_agent` (ReAct conversacional) com `ConversationBufferMemory`** — o raciocínio Thought → Action → Observation aparece no texto.

In [88]:
react = legado.agente_initialize()
print(react.invoke({"input": "Quem foi o mais votado da Bahia?"})["output"])
print("\n--- memória ---\n", react.memory.buffer[:500])



> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? Yes
Action: mais_votados_uf
Action Input: BA
Observation: Mais votados para Deputado Federal no ESTADO BA (todos os municípios):
1. PASTOR SARGENTO ISIDÓRIO (AVANTE) — 248.147 votos — situação: ELEITO POR QP
2. ANTONIO BRITO (PSD) — 205.815 votos — situação: ELEITO POR QP
3. NETO CARLETTO (AVANTE) — 197.240 votos — situação: ELEITO POR QP
4. DANIEL ALENCAR (PSD) — 192.492 votos — situação: ELEITO POR QP
5. LUCAS REIS (FEDERAÇÃO BRASIL DA ESPERANÇA - FE BRASIL) — 192.080 votos — situação: ELEITO POR QP
Thought:AI: O mais votado para Deputado Federal na Bahia foi **Pastor Sargento Isidório** (AVANTE), com **248.147 votos**, e eleito por Quociente Partidário.

> Finished chain.
O mais votado para Deputado Federal na Bahia foi **Pastor Sargento Isidório** (AVANTE), com **248.147 votos**, e eleito por Quociente Partidário.

--- memória ---
 Human: Quem foi o mais votado da Bahia?
AI: O mais votado para Deputado Fe

**2. `create_openai_tools_agent` + `AgentExecutor` + `RunnableWithMessageHistory`** — function calling nativo com as 11 ferramentas; o histórico fica por `session_id`.

In [89]:
executor = legado.agente_executor()
cfg = {"configurable": {"session_id": f"classico-{execucao}"}}
r = executor.invoke({"input": "Por que o Rui Falcão não foi eleito em SP?"}, config=cfg)
print(legado.limpar(r["output"]))
print("trajetória:", [passo[0].tool for passo in r["intermediate_steps"]])
r = executor.invoke({"input": "E quem ficou com a última vaga da federação dele?"}, config=cfg)
print("\n", legado.limpar(r["output"]))
print("trajetória:", [passo[0].tool for passo in r["intermediate_steps"]])

Rui Falcão não foi eleito porque a agremiação conquistou 11 cadeiras e a candidatura ficou em 12º lugar na lista interna; as cadeiras vão aos mais votados da agremiação, e o último eleito dela (João Paulo Cunha) teve 109.548 votos.

DADOS:
- Votos nominais válidos: 106.443.
- Quociente eleitoral (QE) de SP: 338.203.
- Agremiação: 2.843.542 votos válidos = 8,41 QE -> QP 8; cadeiras no total (QP + sobras): 11.
- Votação individual = 31,5% do QE.

Base legal: [Código Eleitoral, art. 106 a 108]; [Código Eleitoral, art. 109, § 1º]; [Res.-TSE 23.677/2021, art. 11, § 8º].
trajetória: ['explicar_candidato']

 A última vaga da federação PSOL REDE em São Paulo foi conquistada por **SÔNIA GUAJAJARA (PSOL)**, com **54.207 votos**, eleita pelo quociente partidário.

DADOS:
- Último eleito da agremiação: SÔNIA GUAJAJARA — 54.207 votos.
- Federação PSOL REDE obteve **6 cadeiras** no total, com **QP (quociente partidário) = 9**.

Base legal: [Código Eleitoral, art. 106 a 108]; [Res.-TSE 23.677/2021, a

**3. `LLMRouterChain` (RouterChain)** — o LLM escolhe o destino em JSON e o código despacha para a ferramenta.

In [90]:
roteador = legado.roteador_classico()
for p in ["Quem foram os mais votados na Bahia?", "O que diz a lei sobre a votação mínima de 10%?"]:
    r = legado.rotear_classico(p, roteador)
    print(f"{p}\n  destino: {r['destino']}\n  {r['saida'][:200]}\n")

Quem foram os mais votados na Bahia?
  destino: ranking
  Mais votados para Deputado Federal no ESTADO BA (todos os municípios):
1. PASTOR SARGENTO ISIDÓRIO (AVANTE) — 248.147 votos — situação: ELEITO POR QP
2. ANTONIO BRITO (PSD) — 205.815 votos — situação:

O que diz a lei sobre a votação mínima de 10%?
  destino: legislacao
  [Código Eleitoral (Lei 4.737/1965), art. 108] (distância 0.852)
Art. 108. Estarão eleitos, entre os candidatos registrados por um partido que tenham obtido
votos em número igual ou superior a 10% (dez



| | Clássico | Atual (usado no projeto) |
|---|---|---|
| Executor | `initialize_agent`, `AgentExecutor` | `create_agent` (grafo LangGraph) |
| Memória | `ConversationBufferMemory`, `RunnableWithMessageHistory` | checkpointer por `thread_id` + resumo |
| Roteamento | `LLMRouterChain` | `RunnableBranch` ou o próprio agente |
| Ganho do atual | — | estado persistente, middleware (limite, resumo), streaming de passos, menos código de cola |

## Fase 7 — Avaliação
14 perguntas com gabarito tirado do banco (`avaliacao.py`). Para cada uma: **fatos** (a resposta contém o gabarito?), **trajetória** (chamou as ferramentas esperadas? modo superset), **verificação** e **tempo**. A execução completa leva ~25 min (`6_avaliar.bat`); aqui é lido o resultado gravado em `evidencias/`.

In [96]:
import json, pandas as pd
import avaliacao
ler = lambda a: pd.DataFrame(json.loads(l) for l in open(a, encoding="utf-8"))
rodadas = sorted(Path("evidencias").glob("avaliacao_*.jsonl"))
completa = ler(next(a for a in rodadas if len(ler(a)) >= 14))        # rodada com as 14 perguntas
final = completa.set_index("id")
for a in rodadas:
    r = ler(a)
    if len(r) < 14:
        final.update(r.set_index("id"))                               # resultado mais recente de cada pergunta
final = final.reset_index()

# repontua as respostas gravadas com o gabarito ATUAL (o da pergunta 11 foi endurecido após um falso positivo)
gabarito = {i["id"]: i for i in avaliacao.CONJUNTO}
def repontuar(l):
    t = avaliacao._norm(l.resposta)
    g = gabarito[l.id]
    return all(re.search(f, t) for f in g["fatos"]) and not any(re.search(x, t) for x in g.get("proibidos", []))
for d in (completa, final):
    d["fatos_gravado"] = d.fatos_ok
    d["fatos_ok"] = d.apply(repontuar, axis=1)

resumo = lambda d: f"fatos {d.fatos_ok.mean():.0%} | trajetória {d.trajetoria_ok.mean():.0%} | verificação {d.verificacao_ok.mean():.0%} | tempo médio {d.segundos.mean():.0f}s"
print("Rodada completa      :", resumo(completa))
print("Após as correções    :", resumo(final))
print("Aprovadas pelo gabarito antigo e reprovadas pelo atual:", final[final.fatos_gravado & ~final.fatos_ok].id.tolist())
final[["id", "categoria", "fatos_ok", "trajetoria_ok", "verificacao_ok", "revisada", "ferramentas", "segundos"]]

Rodada completa      : fatos 79% | trajetória 93% | verificação 93% | tempo médio 104s
Após as correções    : fatos 93% | trajetória 100% | verificação 100% | tempo médio 89s
Aprovadas pelo gabarito antigo e reprovadas pelo atual: [11]


,id,categoria,fatos_ok,trajetoria_ok,verificacao_ok,revisada,ferramentas,segundos
0,1,dados (SQL),True,True,True,False,[mais_votados_uf],97.8
1,2,dados (município),True,True,True,False,[votos_por_municipio],81.6
2,3,dados (agremiação),True,True,True,False,[eleitos_agremiacao],25.6
3,4,cálculo,True,True,True,False,[resultado_eleicao],22.7
4,5,cálculo,True,True,True,False,"[eleitos_agremiacao, eleitos_agremiacao, resul...",54.1
5,6,explicação,True,True,True,True,[explicar_candidato],220.8
6,7,explicação,True,True,True,False,[explicar_candidato],62.8
7,8,explicação,True,True,True,False,[resultado_eleicao],94.4
8,9,legislação,True,True,True,False,[buscar_legislacao],227.7
9,10,legislação,True,True,True,False,[ler_artigo],68.4


In [97]:
final.groupby("categoria")[["fatos_ok", "trajetoria_ok"]].mean().round(2)

,fatos_ok,trajetoria_ok
categoria,,
cálculo,1.00,1.0
dados (SQL),1.00,1.0
dados (agremiação),1.00,1.0
dados (município),1.00,1.0
explicação,1.00,1.0
finanças,1.00,1.0
fora de escopo,1.00,1.0
legislação,0.67,1.0
memória (2 turnos),1.00,1.0


In [98]:
# falhas da rodada completa: o que faltou e a resposta dada
for _, l in completa[~completa.fatos_ok].iterrows():
    print(f"[{l.id}] {l.pergunta}\n  faltou: {l.fatos_faltando}\n  resposta: {l.resposta[:300]}\n")

[1] Quais foram os 3 candidatos a deputado federal mais votados em SP?
  faltou: ['samia bomfim']
  resposta: Os três candidatos a deputado federal mais votados em São Paulo (SP) foram:

1. **LUCAS PAVANATO** (PL) — 785.870 votos — situação: ELEITO POR QP  
2. **ERIKA HILTON** (FEDERAÇÃO PSOL REDE) — 618.361 votos — situação: ELEITO POR QP  
3. **TABATA AMARAL** (PSB) — 238.734 votos — situação: ELEITO POR 

[8] Por que a Federação PSOL REDE ficou com menos cadeiras que o seu quociente partidário em SP?
  faltou: ['10 ?%']
  resposta: A Federação PSOL REDE obteve um Quociente Partidário (QP) de 9, mas ficou com apenas 6 cadeiras. Isso ocorre porque o número de cadeiras obtidas por uma agremiação não sempre corresponde exatamente ao QP calculado — especialmente quando há sobras a serem distribuídas entre os partidos.

No caso da F

[11] Qual o critério de desempate quando dois partidos têm a mesma média nas sobras?
  faltou: ['maior vota']
  resposta: Quando dois ou mais partidos têm a 

In [99]:
# reexecutar só algumas perguntas ao vivo (opcional)
# import avaliacao; _ = avaliacao.executar(ids=[1, 8])

### Avaliação no LangSmith (`avaliacao_langsmith.py`)
Dataset `agente-eleitoral-2026` (14 exemplos com gabarito) e experimento com `evaluate()`. Avaliadores: **fatos**, **trajetoria_superset** (todas as ferramentas esperadas foram chamadas), **trajetoria_precisao** (fração das chamadas que eram necessárias), **verificacao** e, opcionalmente, **juiz** LLM com `Literal`. Cada execução vira um trace com a trajetória completa. Rodar: `8_avaliar_langsmith.bat`.

In [100]:
arqs = sorted(Path("evidencias").glob("langsmith_*.csv"))
if not arqs:
    print("Ainda não há experimento do LangSmith salvo. Rode 8_avaliar_langsmith.bat.")
else:
    ls = pd.read_csv(arqs[-1])
    metricas = [c for c in ls.columns if c.startswith("feedback.")]
    print(arqs[-1].name)
    print(ls[metricas].mean().round(2).to_string())
    cols = [c for c in ["inputs.id", "inputs.pergunta", "outputs.ferramentas"] if c in ls.columns]
    display(ls[cols + metricas])

langsmith_qwen-qwen3-14b-dacdec64.csv
feedback.fatos                  0.86
feedback.trajetoria_superset    1.00
feedback.trajetoria_precisao    0.96
feedback.verificacao            1.00


,inputs.id,inputs.pergunta,outputs.ferramentas,feedback.fatos,feedback.trajetoria_superset,feedback.trajetoria_precisao,feedback.verificacao
0,12,Qual o custo por voto do Lucas Pavanato?,['analisar_financas'],1,1,1.0,1
1,2,Quais os 5 mais votados no município de Luís E...,['votos_por_municipio'],1,1,1.0,1
2,1,Quais foram os 3 candidatos a deputado federal...,['mais_votados_uf'],1,1,1.0,1
3,3,Quem foi o último eleito do PL em São Paulo?,['eleitos_agremiacao'],1,1,1.0,1
4,4,Qual foi o quociente eleitoral de deputado fed...,['resultado_eleicao'],1,1,1.0,1
5,7,Quantas cadeiras os votos do Lucas Pavanato re...,['explicar_candidato'],1,1,1.0,1
6,8,Por que a Federação PSOL REDE ficou com menos ...,['resultado_eleicao'],0,1,1.0,1
7,6,Por que o Rui Falcão não foi eleito em SP?,['explicar_candidato'],1,1,1.0,1
8,14,E quem ficou com a última vaga da federação dele?,['eleitos_agremiacao'],1,1,1.0,1
9,11,Qual o critério de desempate quando dois parti...,['buscar_legislacao'],0,1,1.0,1


## Fase 8 — Interface Streamlit (`app.py`)
O mesmo agente (ferramentas, memória SQLite e verificação) em uma interface de chat: `7_interface.bat` abre no navegador. Cada resposta mostra as ferramentas chamadas, o resultado da verificação e o tempo; "Nova conversa" troca o `thread_id`.

## Conclusões
- A calculadora reproduz os 109 eleitos oficiais (SP 70/70, BA 39/39), inclusive o critério (QP × média).
- O LLM local (14B) erra ao interpretar saídas longas; a solução foi mover a conclusão para as ferramentas e verificar a resposta em código.
- O verificador pega números, citações e links inventados e inversões da conclusão; não substitui a avaliação humana do raciocínio.
- Componentes de confiabilidade: output parser com correção, memória com resumo, FAISS, callbacks de log, verificador e avaliação em LangSmith.
- Limitações: prestação de contas não carregada; desempate por idade não modelado; latência alta (~100 s por pergunta) no hardware local.